Data extraction

In [1]:
import pandas as pd

excel_file = pd.ExcelFile('cs_plot.xlsx')
sheet_names = excel_file.sheet_names

# Create new Excel file
with pd.ExcelWriter('kde_restructured.xlsx', engine='openpyxl') as writer:
    for sheet_name in sheet_names:
        # Read each sheet
        df = pd.read_excel('cs_plot.xlsx', sheet_name=sheet_name)

        # Extract target rows
        target_rows = [148, 440, 731, 1022, 1313, 1604, 1895, 2186, 2477]
        extracted_data = df.iloc[target_rows].copy()

        # Add original row index column
        extracted_data.insert(0, 'Original_Row', [row-2 for row in target_rows])

        # Save to new sheet
        extracted_data.to_excel(writer, sheet_name=sheet_name, index=False)


KDE plot

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from KDEpy import FFTKDE
from KDEpy.bw_selection import silvermans_rule
import warnings
warnings.filterwarnings('ignore')

In [2]:
COLORS = [
    '#0173B2',  # QRLASSOG - blue
    '#DE8F05',  # QRFG - orange
    '#029E73',  # QRNN2G - green
    '#CC78BC',  # MCQRNNG - mauve
    '#CA9161',  # VRLG - brown
    '#949494',  # WRGG - grey
    '#ECA1A6',  # ERMG - pink
    '#56B4E9',  # TSMG - light blue
    '#D55E00',  # TRQG - orange-red
    '#8B0000',  # TRMG - dark red
]

MODEL_NAMES = ['QRLASSOG', 'QRFG', 'QRNN2G', 'MCQRNNG', 'VRLG',
               'WRGG', 'ERMG', 'TSMG', 'TRQG', 'TRMG']

# Publication-quality style settings
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 11,
    'axes.linewidth': 1.0,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
    'axes.unicode_minus': False,
})


# ============================================================================
# KDE computation function (using KDEpy)
# ============================================================================

def calculate_kde(quantile_values, n_points=1000):
    """
    Compute KDE probability density using KDEpy.

    Parameters:
        quantile_values: array of quantile prediction values
        n_points: number of KDE evaluation points

    Returns:
        x, y: x and y coordinates of the KDE curve
    """
    # Normalize data to [0, 1] for stable bandwidth estimation
    pred_min = quantile_values.min()
    pred_max = quantile_values.max()

    if pred_max == pred_min:
        # All values identical: return a point distribution
        return np.array([pred_min]), np.array([1.0])

    pred_norm = (quantile_values - pred_min) / (pred_max - pred_min)

    # Compute optimal bandwidth using Silverman's rule
    len_data = len(pred_norm)
    if len_data > 1:
        tqr = np.percentile(pred_norm, [25, 75])
        IQR = tqr[1] - tqr[0]
        std_val = np.std(pred_norm)

        if std_val > 0 and IQR > 0:
            optimal_bw = 1.06 * min(std_val, IQR / 1.34) * (len_data ** (-1 / 5))
        else:
            optimal_bw = silvermans_rule(pred_norm.reshape(-1, 1))
    else:
        optimal_bw = 0.1

    # Fit and evaluate KDE using FFTKDE
    try:
        kde = FFTKDE(bw=optimal_bw, kernel='gaussian')
        x_norm, y = kde.fit(pred_norm).evaluate(n_points)

        # Inverse-transform back to original scale
        x = x_norm * (pred_max - pred_min) + pred_min

        return x, y
    except Exception as e:
        print(f"KDE computation error: {e}")
        return None, None

In [3]:
def plot_kde_with_actual(kde_file='kde_restructured.xlsx',
                         actual_file='actual.xlsx',
                         output_file='kde_plot.png',
                         dpi=300):
    """
    Plot a 3x3 KDE figure with actual value markers.

    The kde_restructured.xlsx file is structured as:
        - Each sheet = one model (sheet name matches model name)
        - Each row= one time point
        - Columns= 'Original_Row' + quantile values

    Parameters:
        kde_file:    path to KDE data file
        actual_file: path to actual values file (optional; pass None to skip)
        output_file: output PNG filename
        dpi:         image resolution
    """

    print("=" * 70)
    print("KDE probability density plot (using KDEpy)")
    print("=" * 70)

    # Load data
    print("\nLoading data...")
    excel_data = pd.ExcelFile(kde_file)

    # Fix: data structure is one sheet per model, not one column 'Model'.
    # Build a dict {model_name: DataFrame} by reading each matching sheet.
    model_dfs = {}
    for sheet in excel_data.sheet_names:
        if sheet in MODEL_NAMES:
            model_dfs[sheet] = pd.read_excel(kde_file, sheet_name=sheet)

    # Determine number of time points from the first available model
    first_model_df = next(iter(model_dfs.values()))
    n_timepoints = len(first_model_df)
    print(f"Found {len(model_dfs)} models, {n_timepoints} time points")

    # Load actual values if provided
    if actual_file is not None:
        actual_df = pd.read_excel(actual_file)
    else:
        actual_df = None

    # Create figure with grid layout; reserve right margin for the legend
    fig = plt.figure(figsize=(22, 16), dpi=dpi)
    gs = fig.add_gridspec(3, 3, left=0.06, right=0.85, top=0.94, bottom=0.06,
                          hspace=0.25, wspace=0.25)

    axes = []
    for i in range(3):
        for j in range(3):
            axes.append(fig.add_subplot(gs[i, j]))

    print("\nPlotting...")

    # Iterate over each time point (up to 9 subplots)
    for idx in range(min(9, n_timepoints)):
        ax = axes[idx]

        # Read the original row index from the first model's DataFrame
        original_row = first_model_df['Original_Row'].iloc[idx]

        # Retrieve the corresponding actual value
        if actual_df is not None:
            actual_value = actual_df.iloc[original_row, 0]
        else:
            actual_value = None

        # Plot KDE curve for each model
        for model_idx, model_name in enumerate(MODEL_NAMES):
            if model_name not in model_dfs:
                continue

            df = model_dfs[model_name]
            row = df.iloc[idx]

            # All columns except 'Original_Row' are quantile values
            quantile_cols = [col for col in df.columns if col != 'Original_Row']
            quantile_values = row[quantile_cols].values.astype(float)
            quantile_values = quantile_values[~np.isnan(quantile_values)]

            if len(quantile_values) < 2:
                continue

            # Compute KDE
            x, y = calculate_kde(quantile_values, n_points=1000)

            if x is not None and y is not None:
                # TRMG gets a thicker line to stand out as the proposed model
                if model_name == 'TRMG':
                    linewidth, alpha, zorder = 3.0, 0.95, 10
                else:
                    linewidth, alpha, zorder = 2.0, 0.75, 5

                # Only assign a label in the first subplot (shared legend)
                label = model_name if idx == 0 else None

                ax.plot(x, y,
                        color=COLORS[model_idx],
                        linewidth=linewidth,
                        label=label,
                        alpha=alpha,
                        zorder=zorder)

        # Add actual value as a vertical dashed line
        if actual_value is not None:
            ax.axvline(actual_value, color='black', linestyle='--',
                       linewidth=2.5, alpha=0.8, zorder=15,
                       label='Actual Value' if idx == 0 else None)
            title_str = f'Time Point {original_row}\n(Actual: {actual_value:.1f} MW)'
        else:
            title_str = f'Time Point {original_row}'

        # Configure subplot appearance
        ax.set_xlabel('Offshore Wind Power (MW)', fontsize=11)
        ax.set_ylim(bottom=0)
        ax.set_ylabel('Probability Density', fontsize=11)
        ax.set_title(title_str, fontsize=12, fontweight='bold', pad=8)
        ax.grid(True, alpha=0.25, linestyle='--', linewidth=0.5)
        ax.tick_params(labelsize=10)

        # Thin subplot borders
        for spine in ax.spines.values():
            spine.set_linewidth(0.8)

        print(f"  [{idx + 1}/9] Time Point {original_row} done")

    # Place shared legend to the right of the plot area
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels,
               loc='center left',
               bbox_to_anchor=(0.86, 0.5),
               frameon=True,
               framealpha=0.95,
               edgecolor='black',
               fancybox=True,
               fontsize=11,
               title='Models',
               title_fontsize=12)

    # Save figure
    print(f"\nSaving figure...")
    plt.savefig(output_file, dpi=dpi, bbox_inches='tight', format='png')
    print(f"Figure saved: {output_file}")
    print(f"  Resolution: {dpi} DPI")
    print(f"  Format: PNG")
    print(f"  KDE method: KDEpy FFTKDE")

    plt.close()

    print("\n" + "=" * 70)
    print("Plot complete!")
    print("=" * 70)

In [4]:
if __name__ == "__main__":
    # File paths (adjust as needed)
    kde_file = 'kde_restructured.xlsx'
    actual_file = 'actual.xlsx'

    # Generate figure
    plot_kde_with_actual(
        kde_file=kde_file,
        actual_file=actual_file,
        output_file='kde_plot_kdepy.png',
        dpi=400
    )

KDE probability density plot (using KDEpy)

Loading data...
Found 10 models, 9 time points

Plotting...
  [1/9] Time Point 146 done
  [2/9] Time Point 438 done
  [3/9] Time Point 729 done
  [4/9] Time Point 1020 done
  [5/9] Time Point 1311 done
  [6/9] Time Point 1602 done
  [7/9] Time Point 1893 done
  [8/9] Time Point 2184 done
  [9/9] Time Point 2475 done

Saving figure...
Figure saved: kde_plot_kdepy.png
  Resolution: 400 DPI
  Format: PNG
  KDE method: KDEpy FFTKDE

Plot complete!


In [5]:
import gc
gc.collect()

64